1. ติดตั้งไลบรารีที่จำเป็น

In [ ]:
pip install torch transformers scikit-learn pandas numpy datasets

Base Models

In [ ]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# =====================================
#       code สำหรับ โหลด data set
# =====================================

# # โหลดไฟล์ Dataset ของจริงที่คุณทำเสร็จแล้ว
# df = pd.read_csv("thai_emotion_dataset.csv")

# # นำคอลัมน์อารมณ์ต่างๆ มารวมเป็น List [1, 0, 0, 1] ในคอลัมน์เดียว
# label_columns = ['joy', 'sad', 'anger', 'fear'] 
# df['labels'] = df[label_columns].values.tolist()

# print(df.head())
# =====================================

# ==========================================
# 1. จำลองข้อมูล (Dataset) สำหรับ Multi-label
# ==========================================
# สมมติว่าเรามี 4 อารมณ์: [ดีใจ, เศร้า, โกรธ, กลัว]
# ค่า 1 คือมีอารมณ์นั้น, 0 คือไม่มี (1 ประโยคมีได้หลาย 1)
data = {
    "text": [
        "วันนี้ถูกหวย ดีใจมาก แต่ก็กลัวโดนขโมย", # ดีใจ (1), กลัว (1)
        "ฝนตก รถติด หงุดหงิดโว้ยยย",          # โกรธ (1)
        "สอบตกอีกแล้ว ร้องไห้หนักมาก",          # เศร้า (1)
    ],
    "labels": [
        [1, 0, 0, 1], 
        [0, 0, 1, 0], 
        [0, 1, 0, 0]
    ]
}
df = pd.DataFrame(data)
NUM_LABELS = 4 # จำนวนอารมณ์ทั้งหมด

print("ตัวอย่างข้อมูลที่พร้อมเทรน:")
print(df.head())

# ==========================================
# 2. ตั้งค่า Device (ใช้ GPU ถ้ามี เพื่อความเร็ว)
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nกำลังรันบน: {device}")

# ==========================================
# 3. โหลด Tokenizers และ Models 
# ==========================================
# ใช้ AutoModelForSequenceClassification และกำหนด problem_type เป็น "multi_label_classification"

model_names = {
    "wangchan": "airesearch/wangchanberta-base-att-spm-uncased",
    "mbert": "bert-base-multilingual-cased",
    "xlm_roberta": "xlm-roberta-base"
}

tokenizers = {}
models = {}

print("\nกำลังโหลด Models... (อาจใช้เวลาสักครู่ในครั้งแรก)")
for key, path in model_names.items():
    # โหลด Tokenizer (ตัวตัดคำ)
    tokenizers[key] = AutoTokenizer.from_pretrained(path)
    
    # โหลด Model พร้อมกำหนดจำนวน Output Labels
    models[key] = AutoModelForSequenceClassification.from_pretrained(
        path, 
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification" # สำคัญมากสำหรับงาน Multi-label
    ).to(device)
    
print("โหลด Tokenizers และ Models สำเร็จพร้อมลุย!")

ModuleNotFoundError: No module named 'pandas'

ขอโค้ดสเตปต่อไป: วิธีการแปลงข้อความ (Tokenization) และลูปการ Fine-tune โมเดลด้วย PyTorch DataLoader หน่อย
สเตปนี้คือการสร้าง "ตัวจ่ายข้อมูล" (DataLoader) เพื่อแปลงข้อความให้เป็นตัวเลข (Tokenization) และเขียนลูปฝึกสอน (Training Loop) สำหรับ Fine-tune โมเดลครับ

ในการทำจริง คุณต้องรันการ Fine-tune แบบนี้ให้กับโมเดลทั้ง 3 ตัว (WangchanBERTa, mBERT, XLM-RoBERTa) แยกกัน โค้ดด้านล่างนี้จะแสดงโครงสร้างที่สามารถนำไปวนลูปใช้กับโมเดลทุกตัวได้เลย

In [1]:
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from tqdm import tqdm # เอาไว้แสดงแถบความคืบหน้า (Progress bar)

# ==========================================
# 1. สร้าง Dataset Class สำหรับงาน Multi-label
# ==========================================
class EmotionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]

        # แปลงข้อความเป็น Token
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            # บังคับให้ Label เป็น FloatTensor (จำเป็นมากสำหรับ Multi-label / BCE Loss)
            'labels': torch.FloatTensor(label) 
        }

# ==========================================
# 2. ฟังก์ชันสำหรับ Fine-tune โมเดล (ใช้ซ้ำได้กับทุกตัว)
# ==========================================
def train_model(model, dataloader, optimizer, device, epochs=3):
    model.train() # ตั้งค่าโมเดลให้อยู่ในโหมด Training
    
    for epoch in range(epochs):
        total_loss = 0
        progress_bar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        
        for batch in progress_bar:
            # ย้ายข้อมูลไปที่ GPU/CPU
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # เคลียร์ค่า Gradient เก่า
            model.zero_grad()

            # ส่งข้อมูลเข้าโมเดล (Forward pass)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            # ดึงค่า Loss ออกมา (Hugging Face คำนวณ BCEWithLogitsLoss ให้เราแล้ว)
            loss = outputs.loss
            total_loss += loss.item()

            # คำนวณย้อนกลับ (Backward pass) และอัปเดตน้ำหนักโมเดล
            loss.backward()
            optimizer.step()
            
            # อัปเดตแถบความคืบหน้า
            progress_bar.set_postfix({'loss': loss.item()})
            
        avg_loss = total_loss / len(dataloader)
        print(f"จบ Epoch {epoch+1} | Average Loss: {avg_loss:.4f}\n")

# ==========================================
# 3. เริ่มการเทรน (ตัวอย่าง: เทรน WangchanBERTa)
# ==========================================
# ดึงข้อมูลจาก DataFrame ในสเตปที่ 1
texts = df['text'].tolist()
labels = df['labels'].tolist()

# เลือก Tokenizer และ Model ที่ต้องการ
current_tokenizer = tokenizers["wangchan"]
current_model = models["wangchan"]

# เตรียม DataLoader
dataset = EmotionDataset(texts, labels, current_tokenizer, max_len=64)
train_dataloader = DataLoader(dataset, batch_size=2, shuffle=True)

# ตั้งค่า Optimizer (AdamW เป็นมาตรฐานสำหรับ Transformer, เลี่ยง Learning Rate ที่สูงเกินไป)
optimizer = AdamW(current_model.parameters(), lr=2e-5)

print("--- เริ่มการ Fine-tune WangchanBERTa ---")
train_model(current_model, train_dataloader, optimizer, device, epochs=3)

ModuleNotFoundError: No module named 'torch'

สเตป 3: วิธีดึงผลลัพธ์ไปสอน Random Forest